# Phutball MCTS Move Profiler

Profiles where time is spent **within a single MCTS move** for batched self-play.
Breaks down: env step, state conversion, NN forward, legal actions, P2 transform, mctx tree overhead.

Runtime: A100 GPU or TPU v6e

In [ ]:
# Install dependencies
import os
IN_COLAB = 'google.colab' in str(globals()) or os.path.exists('/content')

if IN_COLAB:
    try:
        import jax
        if 'TPU' in str(jax.devices()):
            !pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
        else:
            raise Exception('No TPU')
    except:
        !pip install -q jax[cuda12_pip] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    !pip install -q flax optax mctx

In [ ]:
# Clone repo
REPO_URL = "https://github.com/echoname6/phutball-jax.git"
REPO_DIR = "/content/phutball-jax" if IN_COLAB else "./phutball-jax"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
import jax
import jax.numpy as jnp
import time
from functools import partial

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
DEVICE_TYPE = str(jax.devices()[0]).split(':')[0]
print(f"Using {jax.device_count()}x {DEVICE_TYPE}")

In [ ]:
from phutball_env_jax import (
    EnvConfig, PhutballState, reset, step, get_legal_actions, state_to_network_input,
)
from network import (
    create_network, init_network,
    create_transformer_network, init_transformer_network,
)
from self_play_batched import (
    play_games_batched,
    make_mcts_recurrent_fn,
    batched_mcts_policy,
    batched_reset,
    transformer_mcts_policy,
    make_transformer_recurrent_fn,
    transform_policy_for_p2,
)
import mctx

print("Imports OK")

In [ ]:
# === CONFIG ===
ROWS, COLS = 15, 11
BATCH_SIZE = 64          # games in parallel (matches training)
NUM_SIMS = 32            # MCTS simulations per move

# Transformer config
D_MODEL = 128
N_LAYERS = 6
N_HEADS = 4
FFN_DIM = 256

# CNN config (for comparison)
CNN_CHANNELS = 128
CNN_BLOCKS = 10

env_config = EnvConfig(rows=ROWS, cols=COLS)
ACTION_SPACE = 2 * ROWS * COLS + 1
print(f"Board: {ROWS}x{COLS}, actions: {ACTION_SPACE}")
print(f"Batch: {BATCH_SIZE}, MCTS sims: {NUM_SIMS}")

In [ ]:
# Initialize networks
rng = jax.random.PRNGKey(42)

# Transformer
rng, init_rng = jax.random.split(rng)
tf_net = create_transformer_network(
    rows=ROWS, cols=COLS, d_model=D_MODEL,
    n_layers=N_LAYERS, n_heads=N_HEADS, ffn_dim=FFN_DIM,
)
tf_vars = init_transformer_network(init_rng, tf_net, num_input_channels=6)
tf_params = {'network_params': tf_vars['params']}
tf_nparams = sum(x.size for x in jax.tree_util.tree_leaves(tf_vars['params']))

# CNN
rng, init_rng = jax.random.split(rng)
cnn_net = create_network(rows=ROWS, cols=COLS, num_channels=CNN_CHANNELS, num_res_blocks=CNN_BLOCKS)
cnn_vars = init_network(init_rng, cnn_net, num_input_channels=6)
cnn_params = {
    'network_params': cnn_vars['params'],
    'batch_stats': cnn_vars['batch_stats'],
}
cnn_nparams = sum(x.size for x in jax.tree_util.tree_leaves(cnn_vars['params']))

print(f"Transformer: d={D_MODEL}, {N_LAYERS}L, {N_HEADS}H, ffn={FFN_DIM} — {tf_nparams:,} params")
print(f"CNN:         {CNN_CHANNELS}ch, {CNN_BLOCKS} blocks — {cnn_nparams:,} params")

## Recurrent Function Component Breakdown

Each MCTS simulation calls the recurrent_fn once. It does:
1. **env step** — `vmap(step)(states, actions)`
2. **state convert** — `vmap(state_to_network_input)(states)`
3. **NN forward** — `network.apply(...)`
4. **P2 transform** — `transform_policy_for_p2(logits)`
5. **legal actions** — `vmap(get_legal_actions)(states)`

We time each in isolation at the real batch size, then multiply by `NUM_SIMS` to estimate per-move cost.

In [ ]:
def time_fn(fn, N=200, label=""):
    """Time a function that returns JAX arrays. Returns seconds per call."""
    # Warmup
    out = fn()
    jax.block_until_ready(out)

    start = time.perf_counter()
    for _ in range(N):
        out = fn()
    jax.block_until_ready(out)
    elapsed = time.perf_counter() - start
    per_call = elapsed / N
    if label:
        print(f"  {label:>20s}: {per_call*1000:8.3f} ms")
    return per_call


def profile_recurrent_components(network, params, label, use_transformer):
    """Profile each component of the MCTS recurrent function."""
    print(f"\n{'='*60}")
    print(f"{label} — recurrent_fn components (batch={BATCH_SIZE})")
    print(f"{'='*60}")

    states = batched_reset(env_config, BATCH_SIZE)
    actions = jnp.zeros(BATCH_SIZE, dtype=jnp.int32)

    # JIT each component separately
    @jax.jit
    def do_step(s, a):
        return jax.vmap(lambda st, ac: step(st, ac, env_config))(s, a)

    @jax.jit
    def do_convert(s):
        return jax.vmap(lambda st: state_to_network_input(st, env_config))(s)

    if use_transformer:
        variables = {'params': params['network_params']}
    else:
        variables = {'params': params['network_params'], 'batch_stats': params['batch_stats']}

    @jax.jit
    def do_nn(x):
        return network.apply(variables, x, train=False)

    @jax.jit
    def do_legal(s):
        return jax.vmap(lambda st: get_legal_actions(st, env_config))(s)

    @jax.jit
    def do_p2_transform(logits):
        return transform_policy_for_p2(logits, ROWS, COLS)

    # Pre-compute inputs for downstream steps
    next_states = do_step(states, actions)
    jax.block_until_ready(next_states)
    net_inputs = do_convert(next_states)
    jax.block_until_ready(net_inputs)
    policy_logits, values = do_nn(net_inputs)
    jax.block_until_ready(policy_logits)

    # Time each component
    timings = {}
    timings['env_step'] = time_fn(lambda: do_step(states, actions), label='env step')
    timings['state_convert'] = time_fn(lambda: do_convert(next_states), label='state convert')
    timings['nn_forward'] = time_fn(lambda: do_nn(net_inputs), label='NN forward')
    timings['p2_transform'] = time_fn(lambda: do_p2_transform(policy_logits), label='P2 transform')
    timings['legal_actions'] = time_fn(lambda: do_legal(next_states), label='legal actions')

    recurrent_total = sum(timings.values())
    print(f"  {'recurrent_fn total':>20s}: {recurrent_total*1000:8.3f} ms")

    # Estimated cost per MCTS move = root_eval + NUM_SIMS * recurrent_fn
    root_cost = timings['state_convert'] + timings['nn_forward'] + timings['p2_transform'] + timings['legal_actions']
    estimated_per_move = root_cost + NUM_SIMS * recurrent_total
    print(f"\n  Estimated per move ({NUM_SIMS} sims):")
    print(f"    root eval:          {root_cost*1000:8.3f} ms")
    print(f"    {NUM_SIMS}x recurrent_fn:   {NUM_SIMS * recurrent_total*1000:8.3f} ms")
    print(f"    sum (no tree overhead): {estimated_per_move*1000:8.3f} ms")

    # Percentages within recurrent_fn
    print(f"\n  recurrent_fn breakdown:")
    for k, v in timings.items():
        pct = 100 * v / recurrent_total
        bar = '#' * int(pct / 2)
        print(f"    {k:>15s}: {pct:5.1f}%  {bar}")

    return timings


tf_timings = profile_recurrent_components(tf_net, tf_params, "Transformer", use_transformer=True)
cnn_timings = profile_recurrent_components(cnn_net, cnn_params, "CNN", use_transformer=False)

## Full MCTS Move Timing (end-to-end)

Time the actual `transformer_mcts_policy` / `batched_mcts_policy` call to measure real per-move cost including mctx tree overhead.

In [ ]:
def profile_mcts_move(network, params, policy_fn, recurrent_fn, label, num_calls=20):
    """Time a full MCTS policy call (one move for the whole batch)."""
    print(f"\n{label} — full MCTS move (batch={BATCH_SIZE}, sims={NUM_SIMS}):")

    states = batched_reset(env_config, BATCH_SIZE)
    rng = jax.random.PRNGKey(0)

    # Warmup
    rng, prng = jax.random.split(rng)
    actions, _, _ = policy_fn(
        params, states, prng, network, env_config,
        num_simulations=NUM_SIMS, temperature=1.0,
        recurrent_fn=recurrent_fn,
    )
    actions.block_until_ready()

    # Benchmark
    start = time.perf_counter()
    for _ in range(num_calls):
        rng, prng = jax.random.split(rng)
        actions, _, _ = policy_fn(
            params, states, prng, network, env_config,
            num_simulations=NUM_SIMS, temperature=1.0,
            recurrent_fn=recurrent_fn,
        )
    actions.block_until_ready()
    elapsed = time.perf_counter() - start

    ms_per_move = (elapsed / num_calls) * 1000
    sims_per_sec = (BATCH_SIZE * NUM_SIMS * num_calls) / elapsed
    print(f"  {ms_per_move:.1f} ms/move,  {sims_per_sec:,.0f} sims/sec")
    return ms_per_move


tf_recurrent = make_transformer_recurrent_fn(tf_net, env_config)
cnn_recurrent = make_mcts_recurrent_fn(cnn_net, env_config)

tf_ms = profile_mcts_move(tf_net, tf_params, transformer_mcts_policy, tf_recurrent, "Transformer")
cnn_ms = profile_mcts_move(cnn_net, cnn_params, batched_mcts_policy, cnn_recurrent, "CNN")

## Tree Overhead Estimate

Compare actual MCTS move time against the sum of recurrent_fn components to isolate mctx tree overhead (selection, backprop, Gumbel sampling).

In [ ]:
def show_overhead(timings, actual_ms, label):
    recurrent_total = sum(timings.values())
    root_cost = timings['state_convert'] + timings['nn_forward'] + timings['p2_transform'] + timings['legal_actions']
    estimated_compute_ms = (root_cost + NUM_SIMS * recurrent_total) * 1000
    overhead_ms = actual_ms - estimated_compute_ms
    overhead_pct = 100 * overhead_ms / actual_ms if actual_ms > 0 else 0

    print(f"\n{label}:")
    print(f"  Actual MCTS move:        {actual_ms:8.1f} ms")
    print(f"  Estimated compute:       {estimated_compute_ms:8.1f} ms")
    print(f"  mctx tree overhead:      {overhead_ms:8.1f} ms ({overhead_pct:.1f}%)")

    # Full breakdown including overhead
    components = {
        'NN forward': (root_cost + NUM_SIMS * timings['nn_forward']) * 1000 - root_cost * 1000 + timings['nn_forward'] * 1000,
    }
    # Simpler: show share of actual move time
    nn_ms = (1 + NUM_SIMS) * timings['nn_forward'] * 1000
    step_ms = NUM_SIMS * timings['env_step'] * 1000
    convert_ms = (1 + NUM_SIMS) * timings['state_convert'] * 1000
    legal_ms = (1 + NUM_SIMS) * timings['legal_actions'] * 1000
    p2_ms = (1 + NUM_SIMS) * timings['p2_transform'] * 1000

    print(f"\n  Share of actual move time:")
    parts = [
        ('NN forward', nn_ms),
        ('env step', step_ms),
        ('state convert', convert_ms),
        ('legal actions', legal_ms),
        ('P2 transform', p2_ms),
        ('tree overhead', overhead_ms),
    ]
    for name, ms in parts:
        pct = 100 * ms / actual_ms if actual_ms > 0 else 0
        bar = '#' * int(pct / 2)
        print(f"    {name:>15s}: {ms:7.1f}ms  ({pct:5.1f}%)  {bar}")


show_overhead(tf_timings, tf_ms, "Transformer")
show_overhead(cnn_timings, cnn_ms, "CNN")

## Component Comparison: Transformer vs CNN

In [ ]:
print(f"\n{'Component':<20s} {'Transformer':>12s} {'CNN':>12s} {'Ratio':>8s}")
print("-" * 55)
for k in tf_timings:
    t = tf_timings[k] * 1000
    c = cnn_timings[k] * 1000
    ratio = t / c if c > 0 else float('inf')
    print(f"{k:<20s} {t:>10.3f}ms {c:>10.3f}ms {ratio:>7.2f}x")

print("-" * 55)
print(f"{'MCTS move (actual)':<20s} {tf_ms:>10.1f}ms {cnn_ms:>10.1f}ms {tf_ms/cnn_ms:>7.2f}x")

# Training estimate
moves_per_game = 170
train_batch = 256
games_per_iter = train_batch  # one round of self-play
total_moves = games_per_iter * moves_per_game
# Scale ms/move by batch ratio (profiled at BATCH_SIZE, training at train_batch)
# This is approximate — real scaling depends on hardware utilization
tf_iter_s = total_moves * (tf_ms / 1000) * (BATCH_SIZE / train_batch)
cnn_iter_s = total_moves * (cnn_ms / 1000) * (BATCH_SIZE / train_batch)
print(f"\nEstimated self-play per iteration ({train_batch} games, {NUM_SIMS} sims, ~{moves_per_game} moves/game):")
print(f"  Transformer: {tf_iter_s:.0f}s ({tf_iter_s/60:.1f}min)")
print(f"  CNN:         {cnn_iter_s:.0f}s ({cnn_iter_s/60:.1f}min)")

## NN Forward at Different Batch Sizes

Check if the NN is saturating the GPU at the current batch size.

In [ ]:
print(f"\n{'batch':>6s}  {'Transformer':>12s}  {'CNN':>12s}  {'TF samples/s':>14s}  {'CNN samples/s':>14s}")
print("-" * 65)

rng = jax.random.PRNGKey(0)
tf_apply_vars = {'params': tf_params['network_params']}

for bs in [16, 32, 64, 128, 256, 512]:
    x = jax.random.normal(rng, (bs, 6, ROWS, COLS))

    @jax.jit
    def tf_fwd(x):
        return tf_net.apply(tf_apply_vars, x, train=False)

    @jax.jit
    def cnn_fwd(x):
        return cnn_net.apply(cnn_vars, x, train=False)

    tf_t = time_fn(lambda: tf_fwd(x), N=100)
    cnn_t = time_fn(lambda: cnn_fwd(x), N=100)

    print(f"{bs:>6d}  {tf_t*1000:>10.3f}ms  {cnn_t*1000:>10.3f}ms  {bs/tf_t:>14,.0f}  {bs/cnn_t:>14,.0f}")